# convT-kernel-axis-swap — ex1: compare Conv2d and ConvTranspose2d weight shapes

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `convT-kernel-axis-swap`. Running the final beacon cell reports progress against the `CNN: ConvT kernel axis swap` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `CNN: ConvT kernel axis swap` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`convT-kernel-axis-swap`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "convT-kernel-axis-swap"
DD_SUBTOPIC = "CNN: ConvT kernel axis swap"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## ConvTranspose2d kernel axis swap — quick refresher

`nn.ConvTranspose2d(in_channels=IC, out_channels=OC, kernel_size=(KH, KW)).weight` has shape **`(IC, OC, KH, KW)`** — input channels first, output channels second. This is the **opposite** of `nn.Conv2d`, which uses `(OC, IC, KH, KW)`.

**Why the swap.** ConvTranspose is conv's *adjoint*. Whereas a forward conv contracts input channels into output channels (one filter per OC), the adjoint contracts output channels back into input channels — so the natural axis order flips. PyTorch's storage layout follows that mathematical role: the first axis indexes the *contraction-input* of the operation.

**The trap.** Building a ConvTranspose by copying a Conv2d's `(OC, IC, KH, KW)` weight tensor in won't shape-check the way you expect. You either need to `.transpose(0, 1)` the weight or you'll get a `size mismatch` error.

**Quick check.** `nn.Conv2d(3, 16, 5).weight.shape == (16, 3, 5, 5)`; `nn.ConvTranspose2d(3, 16, 5).weight.shape == (3, 16, 5, 5)`. Memorize the asymmetry — it's a common interview question for the same reason.

### Exercise 1 — compare Conv2d and ConvTranspose2d weight shapes

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Analyze
> LO: Analyze the axis-order asymmetry between `nn.Conv2d` `(OC, IC, KH, KW)` and `nn.ConvTranspose2d` `(IC, OC, KH, KW)` weight tensors by extracting each module's `weight.shape` and labeling each axis.
> Keywords: ConvTranspose2d, weight-shape, axis-swap, introspection
> ```

**KCs targeted:** `convT-weight-axis-order`, `conv-vs-convT-layout`

Implement `ex1_conv_layout_compare(ic, oc, k)`. Construct both a `nn.Conv2d(ic, oc, k)` and a `nn.ConvTranspose2d(ic, oc, k)` with matching constructor args, then return a dict:

```python
{
  'conv_weight_shape':        tuple(...),   # from nn.Conv2d
  'convT_weight_shape':       tuple(...),   # from nn.ConvTranspose2d
  'conv_axis0_role':          'OC' or 'IC',
  'convT_axis0_role':         'OC' or 'IC',
  'axes_swapped':             True | False,
}
```

**The point of the drill.** Both modules take the same `(in_channels, out_channels, kernel_size)` constructor args, yet produce weight tensors whose **first two axes are swapped**. Read both `.weight.shape` tuples directly and label each module's axis-0 role.

**Hint.** `nn.Conv2d(ic, oc, k).weight.shape == (oc, ic, k, k)` (OC first). `nn.ConvTranspose2d(ic, oc, k).weight.shape == (ic, oc, k, k)` (IC first). So:
- `conv_axis0_role = 'OC'`
- `convT_axis0_role = 'IC'`
- `axes_swapped = True` (always, unless `ic == oc`).

In [ ]:
def ex1_conv_layout_compare(ic: int, oc: int, k: int) -> dict:
    from torch import nn
    conv  = nn.Conv2d(ic, oc, k)
    convT = nn.ConvTranspose2d(ic, oc, k)
    return {
        'conv_weight_shape':  tuple(conv.weight.shape),
        'convT_weight_shape': tuple(convT.weight.shape),
        'conv_axis0_role':    'OC',
        'convT_axis0_role':   'IC',
        'axes_swapped':       True,
    }


<details><summary>Solution</summary>

```python
def ex1_conv_layout_compare(ic: int, oc: int, k: int) -> dict:
    from torch import nn
    conv  = nn.Conv2d(ic, oc, k)
    convT = nn.ConvTranspose2d(ic, oc, k)
    return {
        'conv_weight_shape':  tuple(conv.weight.shape),
        'convT_weight_shape': tuple(convT.weight.shape),
        'conv_axis0_role':    'OC',
        'convT_axis0_role':   'IC',
        'axes_swapped':       True,
    }
```

**Why the swap exists.** ConvTranspose is the *adjoint* of convolution. The forward conv contracts IC into OC; its adjoint contracts OC back into IC. PyTorch's storage convention is *first axis = the operation's contraction-input* — so Conv2d's axis 0 is OC (the filter index) and ConvTranspose2d's axis 0 is IC (since transposing the operation also transposes the role of the leading axis).

**The trap in practice.** Loading a Conv2d's `state_dict` into a ConvTranspose2d's slot raises a `size mismatch` error (unless `ic == oc`). The fix is `weight = saved_weight.transpose(0, 1)`. Some checkpoint-converter tools do this automatically — but the asymmetry is the most-cited PyTorch interview gotcha for a reason.

**Why we still call it `axes_swapped=True` when `ic == oc`.** The two tensors have *identical* observable shapes when `ic == oc`, but their **semantic axis roles still differ**. If you treated them as interchangeable and later changed the channel count, the model would silently break — so the role label is the load-bearing fact, not the shape tuple.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()